# 기업 전용 채용 AI - 간단 테스트 프로토타입

OpenAI를 활용한 지원자 추천 및 데이터 분석 시스템

## 1. 환경 설정 및 라이브러리 설치

In [ ]:
# 필요한 패키지 설치
!pip install openai pandas python-dotenv

In [ ]:
import os
import json
import pandas as pd
from openai import OpenAI
from typing import List, Dict
from datetime import datetime, timedelta
import random

In [ ]:
# OpenAI API 키 설정
OPENAI_API_KEY = "your-api-key-here"  # 여기에 실제 API 키를 입력하세요
client = OpenAI(api_key=OPENAI_API_KEY)

## 2. 샘플 데이터 생성

In [ ]:
# 샘플 지원자 데이터
applicants_data = [
    {
        "id": 1,
        "name": "김철수",
        "experience_years": 5,
        "skills": ["Python", "Django", "PostgreSQL", "AWS"],
        "education": "컴퓨터공학 학사",
        "current_position": "백엔드 개발자",
        "introduction": "5년차 백엔드 개발자로 Django를 이용한 RESTful API 개발 경험이 풍부합니다."
    },
    {
        "id": 2,
        "name": "이영희",
        "experience_years": 3,
        "skills": ["React", "TypeScript", "Next.js", "TailwindCSS"],
        "education": "정보통신공학 학사",
        "current_position": "프론트엔드 개발자",
        "introduction": "사용자 경험을 중시하는 3년차 프론트엔드 개발자입니다. React 생태계에 능숙합니다."
    },
    {
        "id": 3,
        "name": "박민수",
        "experience_years": 7,
        "skills": ["Java", "Spring Boot", "Kubernetes", "Docker", "MySQL"],
        "education": "소프트웨어공학 석사",
        "current_position": "시니어 백엔드 개발자",
        "introduction": "마이크로서비스 아키텍처 설계 및 구현 경험이 있는 7년차 개발자입니다."
    },
    {
        "id": 4,
        "name": "최지은",
        "experience_years": 2,
        "skills": ["Python", "TensorFlow", "PyTorch", "Pandas", "Scikit-learn"],
        "education": "데이터사이언스 석사",
        "current_position": "주니어 데이터 사이언티스트",
        "introduction": "머신러닝 모델 개발 및 데이터 분석에 열정이 있는 2년차 데이터 사이언티스트입니다."
    },
    {
        "id": 5,
        "name": "정수현",
        "experience_years": 4,
        "skills": ["Flutter", "Dart", "Firebase", "REST API"],
        "education": "모바일공학 학사",
        "current_position": "모바일 앱 개발자",
        "introduction": "크로스플랫폼 앱 개발 경험이 있는 4년차 모바일 개발자입니다."
    }
]

# DataFrame으로 변환
df_applicants = pd.DataFrame(applicants_data)
print("📋 지원자 데이터:")
df_applicants

In [ ]:
# 샘플 채용공고 데이터
job_posting = {
    "title": "시니어 백엔드 개발자",
    "company": "테크스타트업 주식회사",
    "required_skills": ["Python", "Django", "PostgreSQL", "AWS", "Docker"],
    "experience_required": "5년 이상",
    "description": "대규모 트래픽을 처리하는 백엔드 시스템을 개발할 시니어 개발자를 찾습니다. 클라우드 인프라 경험 우대."
}

print("💼 채용공고:")
print(json.dumps(job_posting, indent=2, ensure_ascii=False))

In [ ]:
# 샘플 광고 성과 데이터 생성
def generate_ad_performance_data():
    dates = [(datetime.now() - timedelta(days=i)).strftime('%Y-%m-%d') for i in range(30, 0, -1)]
    
    data = []
    for date in dates:
        data.append({
            'date': date,
            'impressions': random.randint(500, 2000),
            'clicks': random.randint(20, 150),
            'applications': random.randint(5, 50),
            'cost': round(random.uniform(10000, 50000), 2)
        })
    
    return pd.DataFrame(data)

df_ad_performance = generate_ad_performance_data()
print("📊 광고 성과 데이터 (최근 30일):")
df_ad_performance.head(10)

## 3. OpenAI 기반 핵심 기능 구현

### 3.1 지원자 추천 시스템

In [ ]:
def recommend_candidates(job_posting: Dict, applicants: List[Dict]) -> str:
    """
    채용공고에 맞는 지원자를 추천하고 분석
    """
    prompt = f"""
당신은 HR 전문가입니다. 다음 채용공고에 가장 적합한 지원자를 분석하고 추천해주세요.

📌 채용공고:
- 직무: {job_posting['title']}
- 회사: {job_posting['company']}
- 필수 스킬: {', '.join(job_posting['required_skills'])}
- 경력: {job_posting['experience_required']}
- 상세 설명: {job_posting['description']}

👥 지원자 목록:
{json.dumps(applicants, indent=2, ensure_ascii=False)}

다음 형식으로 분석해주세요:
1. 추천 순위 (TOP 3)
2. 각 추천 지원자별:
   - 적합도 점수 (100점 만점)
   - 강점 3가지
   - 고려사항
   - 면접 시 확인할 질문 2개

명확하고 구체적으로 작성해주세요.
"""
    
    response = client.chat.completions.create(
        model="gpt-4-turbo-preview",
        messages=[
            {"role": "system", "content": "당신은 채용 전문가입니다. 객관적이고 상세한 분석을 제공합니다."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.7,
        max_tokens=2000
    )
    
    return response.choices[0].message.content

In [ ]:
# 지원자 추천 실행
print("🎯 지원자 추천 분석 중...\n")
recommendation = recommend_candidates(job_posting, applicants_data)
print(recommendation)

### 3.2 광고 성과 분석

In [ ]:
def analyze_ad_performance(df: pd.DataFrame) -> str:
    """
    광고 성과 데이터를 분석하고 인사이트 제공
    """
    # 기본 통계
    summary = df.describe().to_string()
    
    # CTR, CVR 계산
    df['ctr'] = (df['clicks'] / df['impressions'] * 100).round(2)
    df['cvr'] = (df['applications'] / df['clicks'] * 100).round(2)
    df['cpa'] = (df['cost'] / df['applications']).round(2)
    
    # 주간 추세
    recent_week = df.tail(7)
    previous_week = df.iloc[-14:-7]
    
    week_comparison = f"""
최근 주간 vs 이전 주간:
- 노출수: {recent_week['impressions'].sum()} vs {previous_week['impressions'].sum()}
- 클릭수: {recent_week['clicks'].sum()} vs {previous_week['clicks'].sum()}
- 지원수: {recent_week['applications'].sum()} vs {previous_week['applications'].sum()}
- 평균 CTR: {recent_week['ctr'].mean():.2f}% vs {previous_week['ctr'].mean():.2f}%
- 평균 CVR: {recent_week['cvr'].mean():.2f}% vs {previous_week['cvr'].mean():.2f}%
"""
    
    prompt = f"""
다음 광고 성과 데이터를 분석하고 인사이트를 제공해주세요.

📊 전체 통계 (30일):
{summary}

📈 주간 비교:
{week_comparison}

📉 최근 7일 상세 데이터:
{recent_week.to_string()}

다음을 분석해주세요:
1. 주요 성과 지표 해석
2. 긍정적인 변화 및 그 이유 (추정)
3. 개선이 필요한 영역
4. 구체적인 액션 아이템 3가지
5. 비용 효율성 평가

마케터가 바로 실행할 수 있는 구체적인 제안을 포함해주세요.
"""
    
    response = client.chat.completions.create(
        model="gpt-4-turbo-preview",
        messages=[
            {"role": "system", "content": "당신은 채용 마케팅 분석 전문가입니다."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.7,
        max_tokens=2000
    )
    
    return response.choices[0].message.content

In [ ]:
# 광고 성과 분석 실행
print("📊 광고 성과 분석 중...\n")
ad_analysis = analyze_ad_performance(df_ad_performance)
print(ad_analysis)

### 3.3 지원자 행동 패턴 분석

In [ ]:
# 샘플 지원자 행동 데이터
applicant_behavior_data = [
    {"job_title": "백엔드 개발자", "applications": 45, "avg_time_on_page": "3분 20초"},
    {"job_title": "프론트엔드 개발자", "applications": 38, "avg_time_on_page": "2분 45초"},
    {"job_title": "데이터 사이언티스트", "applications": 22, "avg_time_on_page": "4분 10초"},
    {"job_title": "DevOps 엔지니어", "applications": 15, "avg_time_on_page": "3분 05초"},
    {"job_title": "프로덕트 매니저", "applications": 31, "avg_time_on_page": "5분 30초"},
]

application_funnel = {
    "공고 조회": 1500,
    "상세 페이지 진입": 850,
    "지원 시작": 320,
    "지원 완료": 151
}

def analyze_applicant_behavior(behavior_data: List[Dict], funnel: Dict) -> str:
    """
    지원자 행동 패턴 분석
    """
    prompt = f"""
다음 지원자 행동 데이터를 분석해주세요.

📊 직무별 지원 현황:
{json.dumps(behavior_data, indent=2, ensure_ascii=False)}

📉 지원 퍼널:
{json.dumps(funnel, indent=2, ensure_ascii=False)}

다음을 분석해주세요:
1. 가장 인기있는 직무 TOP 3와 그 이유 분석
2. 퍼널 각 단계별 이탈률 계산 및 해석
3. 페이지 체류 시간 패턴 분석
4. 이탈률을 낮추기 위한 구체적 개선안 3가지
5. 지원 완료율을 높이기 위한 전략

실행 가능한 제안을 중심으로 작성해주세요.
"""
    
    response = client.chat.completions.create(
        model="gpt-4-turbo-preview",
        messages=[
            {"role": "system", "content": "당신은 사용자 행동 분석 전문가입니다."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.7,
        max_tokens=2000
    )
    
    return response.choices[0].message.content

In [ ]:
# 지원자 행동 패턴 분석 실행
print("👥 지원자 행동 패턴 분석 중...\n")
behavior_analysis = analyze_applicant_behavior(applicant_behavior_data, application_funnel)
print(behavior_analysis)

### 3.4 임베딩 기반 유사도 검색 (보너스)

In [ ]:
import numpy as np
from numpy.linalg import norm

def get_embedding(text: str) -> List[float]:
    """텍스트를 벡터로 변환"""
    response = client.embeddings.create(
        model="text-embedding-3-small",  # 더 저렴한 모델
        input=text
    )
    return response.data[0].embedding

def cosine_similarity(a: List[float], b: List[float]) -> float:
    """코사인 유사도 계산"""
    return np.dot(a, b) / (norm(a) * norm(b))

def find_similar_candidates(job_desc: str, applicants: List[Dict], top_k: int = 3) -> List[Dict]:
    """
    직무 설명과 가장 유사한 지원자 찾기
    """
    # 직무 설명 임베딩
    job_embedding = get_embedding(job_desc)
    
    # 각 지원자 임베딩 및 유사도 계산
    results = []
    for applicant in applicants:
        # 지원자 정보를 텍스트로 변환
        applicant_text = f"""
        이름: {applicant['name']}
        경력: {applicant['experience_years']}년
        기술: {', '.join(applicant['skills'])}
        학력: {applicant['education']}
        현재 직무: {applicant['current_position']}
        소개: {applicant['introduction']}
        """
        
        # 임베딩 및 유사도
        applicant_embedding = get_embedding(applicant_text)
        similarity = cosine_similarity(job_embedding, applicant_embedding)
        
        results.append({
            **applicant,
            'similarity_score': round(similarity * 100, 2)
        })
    
    # 유사도 기준 정렬
    results.sort(key=lambda x: x['similarity_score'], reverse=True)
    
    return results[:top_k]

In [ ]:
# 유사도 검색 실행
print("🔍 임베딩 기반 지원자 검색 중...\n")

job_description = f"{job_posting['title']} - {job_posting['description']} 필요 스킬: {', '.join(job_posting['required_skills'])}"

similar_candidates = find_similar_candidates(job_description, applicants_data, top_k=3)

print("📊 유사도 점수 기반 추천:")
for i, candidate in enumerate(similar_candidates, 1):
    print(f"\n{i}. {candidate['name']} (유사도: {candidate['similarity_score']}점)")
    print(f"   경력: {candidate['experience_years']}년 | 직무: {candidate['current_position']}")
    print(f"   스킬: {', '.join(candidate['skills'])}")

## 4. 대시보드 시각화 (선택)

In [ ]:
# 시각화를 위한 패키지 (선택사항)
!pip install matplotlib seaborn

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8')
plt.rcParams['font.family'] = 'Malgun Gothic'  # 한글 폰트 (Windows)
# Mac의 경우: plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

# 광고 성과 시각화
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 노출수 추이
axes[0, 0].plot(df_ad_performance['date'], df_ad_performance['impressions'], marker='o')
axes[0, 0].set_title('일별 노출수 추이')
axes[0, 0].tick_params(axis='x', rotation=45)
axes[0, 0].grid(True, alpha=0.3)

# 지원수 추이
axes[0, 1].plot(df_ad_performance['date'], df_ad_performance['applications'], marker='o', color='green')
axes[0, 1].set_title('일별 지원수 추이')
axes[0, 1].tick_params(axis='x', rotation=45)
axes[0, 1].grid(True, alpha=0.3)

# CTR 추이
df_ad_performance['ctr'] = (df_ad_performance['clicks'] / df_ad_performance['impressions'] * 100)
axes[1, 0].plot(df_ad_performance['date'], df_ad_performance['ctr'], marker='o', color='orange')
axes[1, 0].set_title('CTR(%) 추이')
axes[1, 0].tick_params(axis='x', rotation=45)
axes[1, 0].grid(True, alpha=0.3)

# 비용 대비 지원수
axes[1, 1].scatter(df_ad_performance['cost'], df_ad_performance['applications'], alpha=0.6)
axes[1, 1].set_xlabel('비용')
axes[1, 1].set_ylabel('지원수')
axes[1, 1].set_title('비용 대비 지원수')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. 종합 리포트 생성

In [ ]:
def generate_comprehensive_report(job_posting: Dict, recommendation: str, ad_analysis: str, behavior_analysis: str) -> str:
    """
    모든 분석 결과를 종합한 리포트 생성
    """
    prompt = f"""
다음 정보를 바탕으로 기업 경영진에게 제출할 종합 리포트를 작성해주세요.

📌 채용공고: {job_posting['title']}

1️⃣ 지원자 추천 결과:
{recommendation}

2️⃣ 광고 성과 분석:
{ad_analysis}

3️⃣ 지원자 행동 패턴:
{behavior_analysis}

다음 형식으로 경영진 리포트를 작성해주세요:
1. Executive Summary (핵심 요약)
2. 주요 발견사항
3. 즉시 실행 가능한 액션 아이템 5가지
4. 예상 효과 및 ROI
5. 리스크 및 고려사항

간결하고 임팩트 있게 작성해주세요.
"""
    
    response = client.chat.completions.create(
        model="gpt-4-turbo-preview",
        messages=[
            {"role": "system", "content": "당신은 전략 컨설턴트입니다. 경영진이 이해하기 쉽고 액션 가능한 리포트를 작성합니다."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.7,
        max_tokens=2500
    )
    
    return response.choices[0].message.content

In [ ]:
# 종합 리포트 생성
print("📄 종합 리포트 생성 중...\n")
print("="*80)
comprehensive_report = generate_comprehensive_report(
    job_posting,
    recommendation,
    ad_analysis,
    behavior_analysis
)
print(comprehensive_report)
print("="*80)

## 6. 결과 저장

In [ ]:
# 결과를 텍스트 파일로 저장
with open('recruitment_ai_report.txt', 'w', encoding='utf-8') as f:
    f.write("=" * 80 + "\n")
    f.write("기업 전용 채용 AI 분석 리포트\n")
    f.write("=" * 80 + "\n\n")
    
    f.write("📊 1. 지원자 추천\n")
    f.write("-" * 80 + "\n")
    f.write(recommendation + "\n\n")
    
    f.write("📈 2. 광고 성과 분석\n")
    f.write("-" * 80 + "\n")
    f.write(ad_analysis + "\n\n")
    
    f.write("👥 3. 지원자 행동 패턴\n")
    f.write("-" * 80 + "\n")
    f.write(behavior_analysis + "\n\n")
    
    f.write("📄 4. 종합 리포트\n")
    f.write("-" * 80 + "\n")
    f.write(comprehensive_report + "\n")

print("✅ 리포트가 'recruitment_ai_report.txt' 파일로 저장되었습니다!")

## 다음 단계

이 프로토타입을 실제 시스템으로 발전시키려면:

1. **데이터베이스 연동**: PostgreSQL + Pinecone으로 실제 데이터 저장
2. **API 서버 구축**: FastAPI로 REST API 개발
3. **인증 시스템**: JWT 기반 기업별 인증
4. **캐싱**: Redis로 자주 묻는 질문 캐싱
5. **프론트엔드**: React로 대시보드 개발
6. **배포**: Docker + AWS/GCP 배포

필요한 부분이 있으면 말씀해주세요!